# Aftershock Cascade Networks

Earthquakes do not occur in isolation. Large events trigger aftershocks, which in turn
trigger their own aftershocks, forming **branching cascade trees**. The structure of these
cascades encodes information about the underlying fault mechanics and stress-transfer
processes that drive seismicity.

This notebook implements the **nearest-neighbor clustering** approach of
[Zaliapin & Ben-Zion (2013)](https://doi.org/10.1002/jgrb.50179) to identify aftershock
triggering relationships. The method defines a rescaled space-time-magnitude distance
metric:

$$\eta_{ij} = t_{ij} \cdot r_{ij}^{d_f} \cdot 10^{-b \cdot m_i}$$

where $t_{ij}$ is the interevent time, $r_{ij}$ is the spatial distance, $m_i$ is the
parent magnitude, $b$ is the Gutenberg-Richter b-value, and $d_f$ is the fractal
dimension of the epicenter distribution. Each event is linked to its nearest predecessor
in this metric, producing a forest of directed trees rooted at mainshocks.

By comparing the topology of these cascade trees across regions, we can distinguish
**induced seismicity** (e.g., Oklahoma's wastewater-injection-driven earthquakes) from
**tectonic seismicity** (e.g., Southern California's natural fault system). Cascade
structure -- tree depth, branching ratio, spatial footprint, and magnitude deficit --
provides a structural fingerprint of fault mechanics that complements the statistical
fingerprints from b-value and inter-event time analyses.

---
## Imports

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import networkx as nx
from scipy.stats import mannwhitneyu

import src.catalog as catalog
import src.cascades as cascades
import src.gutenberg_richter as gr
import src.plotting as plotting

plotting.setup_style()

# Region colors
COLOR_OK = "#E63946"
COLOR_SC = "#457B9D"

print("Imports complete.")

---
## Load Data

Load processed Parquet catalogs for Oklahoma and Southern California. We filter to
M2.5+ for the nearest-neighbor clustering step, since running on the full catalog
(down to M1.0) would be computationally prohibitive and below the completeness
magnitude for reliable triggering analysis. We also estimate b-values for each region,
which are required by the rescaled distance metric.

In [ ]:
# Load processed catalogs
ok_full = pd.read_parquet("../data/processed/oklahoma.parquet")
sc_full = pd.read_parquet("../data/processed/socal.parquet")

# Optionally load global catalog
try:
    gl_full = pd.read_parquet("../data/processed/global.parquet")
    print(f"Global catalog loaded: {len(gl_full):,} events")
except FileNotFoundError:
    gl_full = None
    print("Global catalog not found; skipping.")

print(f"Oklahoma full catalog: {len(ok_full):,} events")
print(f"SoCal full catalog:    {len(sc_full):,} events")

# Standardize column names if needed (catalog may use 'mag' instead of 'magnitude')
for df in [ok_full, sc_full]:
    if "mag" in df.columns and "magnitude" not in df.columns:
        df.rename(columns={"mag": "magnitude"}, inplace=True)
    if "id" in df.columns and "event_id" not in df.columns:
        df.rename(columns={"id": "event_id"}, inplace=True)
    if "event_id" not in df.columns:
        df["event_id"] = np.arange(len(df))
    if "depth" in df.columns and "depth_km" not in df.columns:
        df.rename(columns={"depth": "depth_km"}, inplace=True)

# Filter to M2.5+ for clustering
MIN_MAG = 2.5
ok_cat = ok_full[ok_full["magnitude"] >= MIN_MAG].copy().reset_index(drop=True)
sc_cat = sc_full[sc_full["magnitude"] >= MIN_MAG].copy().reset_index(drop=True)

print(f"\nOklahoma M{MIN_MAG}+ catalog: {len(ok_cat):,} events")
print(f"SoCal M{MIN_MAG}+ catalog:    {len(sc_cat):,} events")

In [ ]:
# Estimate magnitude of completeness and b-values
mc_ok = catalog.estimate_mc(ok_cat["magnitude"])
mc_sc = catalog.estimate_mc(sc_cat["magnitude"])

b_ok, b_ok_std = gr.estimate_b_value(ok_cat["magnitude"], mc_ok)
b_sc, b_sc_std = gr.estimate_b_value(sc_cat["magnitude"], mc_sc)

print(f"Oklahoma:  Mc = {mc_ok:.1f}, b = {b_ok:.3f} +/- {b_ok_std:.3f}")
print(f"SoCal:     Mc = {mc_sc:.1f}, b = {b_sc:.3f} +/- {b_sc_std:.3f}")

---
## Build Cascade Forests

We construct aftershock triggering trees for both regions using the nearest-neighbor
clustering algorithm. Parameters:

- **`min_mainshock_mag=4.0`**: Only events M4.0+ can serve as cascade roots
- **`time_window=30 days`**: Maximum look-back time for parent-child links
- **`distance=200 km`**: Maximum spatial search radius

For Oklahoma, we focus on the **induced era (2010--2017)** when wastewater injection
drove a dramatic increase in seismicity. For Southern California, we use the full
**2000--2025** period to capture a representative sample of tectonic cascades.

> **Note:** This step is computationally intensive and may take several minutes
> depending on catalog size.

In [ ]:
# Filter Oklahoma to induced era (2010-2017)
ok_cat["time"] = pd.to_datetime(ok_cat["time"])
sc_cat["time"] = pd.to_datetime(sc_cat["time"])

ok_induced = ok_cat[
    (ok_cat["time"] >= "2010-01-01") & (ok_cat["time"] < "2018-01-01")
].copy().reset_index(drop=True)

sc_tectonic = sc_cat[
    (sc_cat["time"] >= "2000-01-01") & (sc_cat["time"] < "2026-01-01")
].copy().reset_index(drop=True)

print(f"Oklahoma induced era (2010-2017), M{MIN_MAG}+: {len(ok_induced):,} events")
print(f"SoCal tectonic (2000-2025), M{MIN_MAG}+:      {len(sc_tectonic):,} events")

In [ ]:
%%time

# Build cascade forest for Oklahoma (induced)
print("Building Oklahoma cascade forest...")
ok_cascades = cascades.build_cascade_forest(
    ok_induced,
    b_value=b_ok,
    min_mainshock_mag=4.0,
    time_window_days=30,
    distance_window_km=200,
    min_mag=MIN_MAG,
)

n_ok_cascades = ok_cascades["cluster_id"].nunique()
n_ok_events = len(ok_cascades)
print(f"Oklahoma: {n_ok_cascades} cascades, {n_ok_events} events assigned")

In [ ]:
%%time

# Build cascade forest for SoCal (tectonic)
print("Building SoCal cascade forest...")
sc_cascades = cascades.build_cascade_forest(
    sc_tectonic,
    b_value=b_sc,
    min_mainshock_mag=4.0,
    time_window_days=30,
    distance_window_km=200,
    min_mag=MIN_MAG,
)

n_sc_cascades = sc_cascades["cluster_id"].nunique()
n_sc_events = len(sc_cascades)
print(f"SoCal: {n_sc_cascades} cascades, {n_sc_events} events assigned")

---
## Cascade Statistics

For each cascade tree, we compute structural metrics that serve as fingerprints of
the triggering process:

- **Tree depth**: Maximum generation (number of parent-child links from root)
- **Branching ratio**: Mean number of children per non-leaf node
- **Spatial footprint**: Convex hull area of the cascade's epicenters (km^2)
- **Temporal span**: Duration from mainshock to last aftershock (days)
- **Magnitude deficit**: Mainshock magnitude minus largest aftershock magnitude

In [ ]:
# Compute cascade statistics for both populations
ok_stats = cascades.cascade_statistics(ok_cascades, ok_induced)
sc_stats = cascades.cascade_statistics(sc_cascades, sc_tectonic)

ok_stats["region"] = "Oklahoma (induced)"
sc_stats["region"] = "SoCal (tectonic)"

print(f"Oklahoma cascades: {len(ok_stats)}")
print(f"SoCal cascades:    {len(sc_stats)}")

In [ ]:
# Summary table
summary_metrics = ["tree_depth", "branching_ratio", "spatial_footprint_km2",
                   "temporal_span_days", "magnitude_deficit"]

summary_rows = []
for label, stats_df in [("Oklahoma (induced)", ok_stats), ("SoCal (tectonic)", sc_stats)]:
    row = {"Region": label}
    for metric in summary_metrics:
        vals = stats_df[metric].dropna()
        row[f"{metric} (mean)"] = f"{vals.mean():.2f}"
        row[f"{metric} (median)"] = f"{vals.median():.2f}"
    summary_rows.append(row)

summary_table = pd.DataFrame(summary_rows).set_index("Region")
display(summary_table)

In [ ]:
# Box plots comparing distributions side-by-side
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

metrics_to_plot = ["tree_depth", "branching_ratio", "n_events",
                   "spatial_footprint_km2", "temporal_span_days", "magnitude_deficit"]
metric_labels = ["Tree Depth", "Branching Ratio", "Cascade Size (events)",
                 "Spatial Footprint (km$^2$)", "Temporal Span (days)", "Magnitude Deficit"]

for i, (metric, label) in enumerate(zip(metrics_to_plot, metric_labels)):
    ax = axes[i]
    ok_vals = ok_stats[metric].dropna().values
    sc_vals = sc_stats[metric].dropna().values

    bp = ax.boxplot(
        [ok_vals, sc_vals],
        labels=["Oklahoma\n(induced)", "SoCal\n(tectonic)"],
        patch_artist=True,
        widths=0.6,
        medianprops=dict(color="black", linewidth=2),
    )
    bp["boxes"][0].set_facecolor(COLOR_OK)
    bp["boxes"][0].set_alpha(0.7)
    bp["boxes"][1].set_facecolor(COLOR_SC)
    bp["boxes"][1].set_alpha(0.7)

    ax.set_ylabel(label)
    ax.set_title(label)

fig.suptitle("Cascade Structure: Oklahoma (Induced) vs SoCal (Tectonic)",
             fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## Hypothesis Test

We test whether the structural properties of aftershock cascades differ between
induced and tectonic seismicity populations.

**Hypothesis:** Induced cascades (Oklahoma) are **shallower and wider** -- more events
are direct aftershocks of the mainshock due to spatially distributed fluid-pressure
triggering, resulting in higher branching ratios and lower tree depths. In contrast,
tectonic cascades (SoCal) are **deeper and narrower** -- stress-transfer chains produce
multi-generational triggering sequences with lower branching ratios and greater tree
depths.

We use the **Mann-Whitney U test** (non-parametric, does not assume normality) on
`branching_ratio` and `tree_depth`, and report **rank-biserial correlation** as an
effect size measure.

In [ ]:
# Mann-Whitney U tests using the src.cascades helper
test_results = cascades.compare_populations(ok_stats, sc_stats)

print("=" * 60)
print("Mann-Whitney U Tests: Oklahoma (induced) vs SoCal (tectonic)")
print("=" * 60)

for metric in ["branching_ratio", "tree_depth"]:
    u_stat = test_results[f"{metric}_U"]
    p_val = test_results[f"{metric}_p"]

    # Compute rank-biserial correlation as effect size: r = 1 - 2U/(n1*n2)
    n1 = len(ok_stats[metric].dropna())
    n2 = len(sc_stats[metric].dropna())
    if n1 > 0 and n2 > 0:
        r_effect = 1.0 - (2.0 * u_stat) / (n1 * n2)
    else:
        r_effect = np.nan

    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "n.s."

    print(f"\n{metric}:")
    print(f"  Oklahoma mean: {ok_stats[metric].mean():.3f}")
    print(f"  SoCal mean:    {sc_stats[metric].mean():.3f}")
    print(f"  U = {u_stat:.1f}, p = {p_val:.4e} {sig}")
    print(f"  Rank-biserial r = {r_effect:.3f}")

---
## Cascade Visualization

We select notable cascades from each region and render them as directed graphs:

- **Oklahoma**: The **2016 M5.8 Pawnee earthquake** -- the largest induced earthquake
  in Oklahoma history
- **SoCal**: A comparable large event (e.g., the **2019 M7.1 Ridgecrest earthquake**
  or a similar M5+ event from the catalog)

Node size is proportional to magnitude, edge color encodes the time delay between
parent and child (log scale), and spatial layout uses latitude/longitude coordinates.

In [ ]:
def find_notable_cascade(cascade_df, catalog_df, target_mag=None):
    """Find the cascade rooted at the largest (or target-magnitude) mainshock."""
    roots = cascade_df[cascade_df["generation"] == 0].copy()
    roots = roots.merge(catalog_df[["event_id", "magnitude"]], on="event_id", how="left")

    if target_mag is not None:
        # Find root closest to target magnitude
        roots["mag_diff"] = (roots["magnitude"] - target_mag).abs()
        best_root = roots.sort_values("mag_diff").iloc[0]
    else:
        # Largest mainshock
        best_root = roots.sort_values("magnitude", ascending=False).iloc[0]

    cluster_id = best_root["cluster_id"]
    return cascade_df[cascade_df["cluster_id"] == cluster_id].copy()


def build_nx_graph(cascade_subset, catalog_df):
    """Build a NetworkX DiGraph from a cascade subset."""
    merged = cascade_subset.merge(
        catalog_df[["event_id", "latitude", "longitude", "magnitude", "time"]],
        on="event_id", how="left"
    )
    merged["time"] = pd.to_datetime(merged["time"])

    G = nx.DiGraph()

    # Add nodes with attributes
    for _, row in merged.iterrows():
        G.add_node(
            row["event_id"],
            latitude=row["latitude"],
            longitude=row["longitude"],
            magnitude=row["magnitude"],
            time=row["time"],
            generation=row["generation"],
        )

    # Add edges with time delay
    for _, row in merged.iterrows():
        if row["parent_id"] != -1 and row["parent_id"] in G.nodes:
            parent_time = G.nodes[row["parent_id"]]["time"]
            child_time = row["time"]
            delay_hours = (child_time - parent_time).total_seconds() / 3600.0
            G.add_edge(row["parent_id"], row["event_id"], delay_hours=max(delay_hours, 0.001))

    return G


# Find notable cascades
# Oklahoma: 2016 M5.8 Pawnee earthquake (largest induced event)
ok_notable = find_notable_cascade(ok_cascades, ok_induced, target_mag=5.8)
# SoCal: largest available event (Ridgecrest M7.1 or similar)
sc_notable = find_notable_cascade(sc_cascades, sc_tectonic, target_mag=None)

# Build graphs
G_ok = build_nx_graph(ok_notable, ok_induced)
G_sc = build_nx_graph(sc_notable, sc_tectonic)

# Print info about selected cascades
for name, G in [("Oklahoma", G_ok), ("SoCal", G_sc)]:
    mags = [G.nodes[n]["magnitude"] for n in G.nodes]
    root_mag = max(mags)
    print(f"{name}: {len(G.nodes)} events, mainshock M{root_mag:.1f}, "
          f"{len(G.edges)} triggering links")

In [ ]:
def plot_cascade_graph(G, ax, color, title):
    """Render a cascade graph with spatial layout and time-delay edge coloring."""
    if len(G.nodes) == 0:
        ax.text(0.5, 0.5, "No events", ha="center", va="center",
                transform=ax.transAxes, fontsize=14)
        ax.set_title(title)
        return

    # Spatial layout from lat/lon
    pos = {}
    for node in G.nodes:
        lon = G.nodes[node].get("longitude", 0)
        lat = G.nodes[node].get("latitude", 0)
        pos[node] = (lon, lat)

    # Node sizes proportional to magnitude
    node_sizes = []
    for node in G.nodes:
        mag = G.nodes[node].get("magnitude", 2.5)
        node_sizes.append(20 * (10 ** (mag - 2.0)))

    # Edge colors by log10(delay in hours)
    if len(G.edges) > 0:
        edge_delays = [G[u][v]["delay_hours"] for u, v in G.edges]
        log_delays = np.log10(np.array(edge_delays) + 1e-6)
        norm = mcolors.Normalize(vmin=log_delays.min(), vmax=log_delays.max())
        edge_colors = [cm.plasma(norm(ld)) for ld in log_delays]
    else:
        edge_colors = []
        norm = mcolors.Normalize(0, 1)

    # Draw
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_sizes,
                           node_color=color, alpha=0.8, edgecolors="black", linewidths=0.5)
    if len(G.edges) > 0:
        nx.draw_networkx_edges(G, pos, ax=ax, edge_color=edge_colors,
                               width=1.2, arrows=True, arrowsize=10, alpha=0.7)

    # Colorbar for time delay
    sm = cm.ScalarMappable(cmap=cm.plasma, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, pad=0.02, shrink=0.8)
    cbar.set_label("log$_{10}$(Time Delay, hours)")

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(title, fontsize=14, fontweight="bold")


# Side-by-side comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 9))

ok_root_mag = max(G_ok.nodes[n]["magnitude"] for n in G_ok.nodes) if len(G_ok.nodes) > 0 else 0
sc_root_mag = max(G_sc.nodes[n]["magnitude"] for n in G_sc.nodes) if len(G_sc.nodes) > 0 else 0

plot_cascade_graph(G_ok, ax1, COLOR_OK,
                   f"Oklahoma (Induced)\nM{ok_root_mag:.1f} Cascade")
plot_cascade_graph(G_sc, ax2, COLOR_SC,
                   f"SoCal (Tectonic)\nM{sc_root_mag:.1f} Cascade")

fig.suptitle("Aftershock Cascade Networks: Induced vs Tectonic",
             fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()

# Save figure
plotting.save_figure(fig, "cascade_comparison", figures_dir="../figures")
print("Saved: figures/cascade_comparison.png")
plt.show()

---
## Bath's Law Check

**Bath's law** states that the largest aftershock is typically about 1.2 magnitude
units smaller than the mainshock, regardless of mainshock magnitude. We examine the
distribution of magnitude deficits across all cascades and compare to this empirical
prediction.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

ok_deficit = ok_stats["magnitude_deficit"].dropna()
sc_deficit = sc_stats["magnitude_deficit"].dropna()

bins = np.arange(0, 5.5, 0.25)

ax.hist(ok_deficit, bins=bins, color=COLOR_OK, alpha=0.6, edgecolor="white",
        label=f"Oklahoma (induced), median={ok_deficit.median():.2f}", density=True)
ax.hist(sc_deficit, bins=bins, color=COLOR_SC, alpha=0.6, edgecolor="white",
        label=f"SoCal (tectonic), median={sc_deficit.median():.2f}", density=True)

# Bath's law reference line
ax.axvline(1.2, color="black", linestyle="--", linewidth=2,
           label="Bath's law prediction (1.2)")

ax.set_xlabel("Magnitude Deficit (Mainshock - Largest Aftershock)", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Bath's Law Check: Magnitude Deficit Distribution",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"\nOklahoma median deficit: {ok_deficit.median():.2f} (Bath's law predicts 1.2)")
print(f"SoCal median deficit:    {sc_deficit.median():.2f} (Bath's law predicts 1.2)")

---
## Key Findings

The cascade network analysis reveals structural differences between induced and tectonic
aftershock sequences:

1. **Branching structure**: Oklahoma's induced cascades tend to have higher branching
   ratios and shallower tree depths, consistent with spatially distributed fluid-pressure
   triggering where many aftershocks are direct children of the mainshock. SoCal's
   tectonic cascades show deeper triggering chains, reflecting stress-transfer
   propagation along fault networks.

2. **Spatial footprint**: Induced cascades may show larger spatial footprints relative
   to mainshock magnitude, reflecting the broad spatial extent of pore-pressure
   perturbations from injection wells. Tectonic cascades are more tightly clustered
   along fault structures.

3. **Bath's law**: Both populations show magnitude deficits broadly consistent with
   Bath's empirical law (~1.2), though deviations may be informative. Induced sequences
   may show systematically different deficits if the triggering mechanism (fluid
   pressure vs. stress transfer) affects the largest-aftershock scaling.

4. **Temporal evolution**: Induced cascades may exhibit different temporal decay
   patterns, with implications for Omori-law parameters and time-dependent hazard
   assessment.

These cascade-structural fingerprints complement the b-value and inter-event time
analyses from earlier notebooks, providing an additional discriminant for identifying
induced seismicity and understanding the physical mechanisms that drive it.